# Lecture 15: Object Detection

*Notebook for UMich EECS 498/598 (Justin Johnson), Lecture 15 "Object Detection" (Fall 2020 slides), with material from Stanford CS231n 2026, Lecture 9 "Detection, Segmentation, Visualization, and Understanding".*

So far every network answered one question: **what** is in this image? **Object detection** also asks **where**, for **every** object.

**What you'll learn**
1. The detection task, and how to find a **single** object (classification + box regression).
2. Why many objects are hard: **sliding windows** and **region proposals**.
3. The R-CNN family: **R-CNN → Fast R-CNN → Faster R-CNN**, plus **single-stage** detectors like YOLO.
4. The tools every detector needs: **IoU**, **non-max suppression**, and **mean Average Precision (mAP)**.
5. **Build and train a tiny detector** from scratch, and evaluate it with mAP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. Computer vision tasks

| Task | Output |
|---|---|
| **Classification** | one label for the whole image ("cat") |
| **Semantic segmentation** | a label for every pixel (grass, cat, tree, sky); no separate objects |
| **Object detection** | a label **and a box** for every object (dog, dog, cat) |
| **Instance segmentation** | a label and the exact pixels of every object |

Today: **object detection**. Next lecture: segmentation.

### The task
* **Input:** one RGB image.
* **Output:** a **set** of detected objects. For each one: a **category label** (from a fixed set) and a **bounding box**: 4 numbers, e.g. $(x, y, \text{width}, \text{height})$.

**Why is it harder than classification?**
* **Multiple outputs:** each image has a different number of objects.
* **Two kinds of output:** "what" (a category) and "where" (a box).
* **Large images:** classification works at 224×224; detection needs more resolution, often around 800×600.

## 2. Detecting a single object

If we knew there was exactly **one** object, we could reuse a classification network and add a second head:
* **"What":** a fully connected layer → class scores → **softmax loss**.
* **"Where":** another fully connected layer → 4 box numbers → **L2 loss** against the correct box. ("Treat localization as a regression problem!")

The two losses are combined into one **multitask loss**: a **weighted sum**. The backbone is usually **pretrained on ImageNet** (transfer learning, Lecture 11).

Here's the multitask loss for one image, by hand:

In [ ]:
class_scores = np.array([2.0, 0.5, -1.0])            # scores for (cat, dog, car)
correct_class = 0                                     # it's a cat
predicted_box = np.array([50, 40, 100, 80])           # (x, y, w, h)
correct_box = np.array([55, 38, 90, 85])

probs = np.exp(class_scores) / np.exp(class_scores).sum()
softmax_loss = -np.log(probs[correct_class])
l2_loss = np.sum((predicted_box - correct_box) ** 2)
weight = 0.001                                        # a hyperparameter: how much to care about the box
total = softmax_loss + weight * l2_loss
print(f"softmax loss {softmax_loss:.3f} + {weight} x L2 loss {l2_loss} = multitask loss {total:.3f}")

Choosing the weight between the losses is a hyperparameter you have to tune: the two losses have different units and scales.

**Problem:** images can contain **more than one** object, and a different number each time (4 numbers for one cat, 12 for three animals, many for a crowd of ducks). A fixed-size output layer can't do that.

## 3. Multiple objects: sliding windows

**Idea:** apply a CNN to many different **crops** of the image, and classify each crop as an object category or **background**.

**Question: how many possible boxes are there in an image of size H × W?** A box of size $h\times w$ can be placed at $(W - w + 1)$ horizontal and $(H - h + 1)$ vertical positions. Summing over all box sizes:

$$\sum_{h=1}^{H}\sum_{w=1}^{W}(W - w + 1)(H - h + 1) = \frac{H(H+1)}{2}\cdot\frac{W(W+1)}{2}$$

In [ ]:
def count_boxes(H, W):
    return (H * (H + 1) // 2) * (W * (W + 1) // 2)

print(f"tiny 4x4 image:  {count_boxes(4, 4):,} boxes")
print(f"800x600 image:   {count_boxes(600, 800):,} boxes  (about {count_boxes(600, 800) / 1e9:.0f} BILLION)")

(The slide says "~58M", but the formula on the same slide gives about **58 billion**. Either way: there's no way we can run a CNN on all of them.)

## 4. Region proposals

**Find a small set of boxes that are likely to cover all the objects**, using a fast heuristic (e.g., look for "blob-like" regions). **Selective Search** gives about **2,000** region proposals in a few seconds on a CPU. Then we only need to classify those 2,000 boxes instead of billions.

## 5. R-CNN ("Region-based CNN", Girshick et al., 2014)

1. Run a region proposal method: about **2,000 regions of interest (RoIs)**.
2. **Warp** (resize) each region to 224×224.
3. Run **each region** through a CNN, which predicts **class scores** and a **box correction**.
4. Use the scores to choose which boxes to output.

### Box regression: correcting the proposal
The proposal is probably not exactly right, so the CNN predicts a **transform** $(t_x, t_y, t_w, t_h)$ that corrects it. With the proposal given by its center and size $(p_x, p_y, p_w, p_h)$:

$$b_x = p_x + p_w\,t_x \qquad b_y = p_y + p_h\,t_y \qquad b_w = p_w\,e^{t_w} \qquad b_h = p_h\,e^{t_h}$$

* The shift is **relative to the box size** (so it means the same thing for big and small boxes).
* The size change is in **log space** (so the box can never get a negative width).

In [ ]:
def apply_transform(proposal, t):
    # proposal and output: (center x, center y, width, height)
    px, py, pw, ph = proposal
    tx, ty, tw, th = t
    return np.array([px + pw * tx, py + ph * ty, pw * np.exp(tw), ph * np.exp(th)])

def transform_target(proposal, true_box):
    # The transform that would turn `proposal` into `true_box` exactly: what the network is trained to predict.
    px, py, pw, ph = proposal
    bx, by, bw, bh = true_box
    return np.array([(bx - px) / pw, (by - py) / ph, np.log(bw / pw), np.log(bh / ph)])

proposal = np.array([100, 80, 50, 40])
true_box = np.array([110, 76, 60, 30])
t = transform_target(proposal, true_box)
print("training target t =", t.round(3))
print("applying it to the proposal gives:", apply_transform(proposal, t), "(= the true box)")

Before we look at faster versions of R-CNN, we need three tools that **every** detector uses.

## 6. Comparing boxes: Intersection over Union (IoU)

How do we compare a predicted box with the ground-truth box?

$$\text{IoU} = \frac{\text{area of intersection}}{\text{area of union}}$$

(Also called the "Jaccard index".) Rules of thumb from the lecture: **IoU > 0.5 is "decent", > 0.7 is "pretty good", > 0.9 is "almost perfect".**

We'll store boxes as corners, $(x_1, y_1, x_2, y_2)$, which makes the intersection easy:

In [ ]:
def iou(box_a, box_b):
    # Boxes are (x1, y1, x2, y2).
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)     # 0 if the boxes don't overlap
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    return intersection / (area_a + area_b - intersection)

truth = (10, 10, 50, 50)
for name, guess in [("shifted a lot", (25, 20, 65, 60)), ("shifted a little", (14, 12, 54, 52)), ("almost exact", (11, 10, 51, 50))]:
    print(f"{name:16s}: IoU = {iou(truth, guess):.2f}")

## 7. Overlapping boxes: non-max suppression (NMS)

Detectors usually output **many overlapping boxes** for the same object. **Non-max suppression** cleans them up:
1. Select the **highest-scoring** remaining box.
2. **Remove** lower-scoring boxes that overlap it with IoU > a threshold (e.g., 0.7).
3. If any boxes remain, go back to step 1.

The lecture's example: two dogs, each detected twice.

In [ ]:
def nms(boxes, scores, threshold=0.7):
    order = list(np.argsort(scores)[::-1])          # highest score first
    keep = []
    while order:
        best = order.pop(0)                           # 1. take the best remaining box
        keep.append(best)
        order = [i for i in order if iou(boxes[best], boxes[i]) <= threshold]   # 2. drop boxes that overlap it
    return keep

dog_boxes = [(10, 20, 60, 70), (14, 22, 62, 74),       # two boxes on dog #1
             (70, 25, 120, 80), (73, 22, 122, 76)]     # two boxes on dog #2
dog_scores = np.array([0.9, 0.75, 0.8, 0.7])

print("IoU between the two boxes on dog #1:", round(iou(dog_boxes[0], dog_boxes[1]), 2))
print("IoU between the two boxes on dog #2:", round(iou(dog_boxes[2], dog_boxes[3]), 2))
print("IoU between boxes on different dogs:", round(iou(dog_boxes[0], dog_boxes[2]), 2))
kept = nms(dog_boxes, dog_scores)
print("NMS keeps the boxes with scores:", [float(dog_scores[i]) for i in kept])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, which, title in [(axes[0], range(4), "before NMS"), (axes[1], kept, "after NMS")]:
    for i in which:
        x1, y1, x2, y2 = dog_boxes[i]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=f"C{i}", lw=2))
        ax.text(x1, y1 - 2, f"P(dog) = {dog_scores[i]}", color=f"C{i}", fontsize=8)
    ax.set_xlim(0, 130); ax.set_ylim(90, 0); ax.set_title(title); ax.set_aspect("equal")
plt.show()

**A known problem:** when objects really do overlap a lot (a crowd of people), NMS may delete correct boxes. There's no perfect solution.

## 8. Evaluating detectors: mean Average Precision (mAP)

Accuracy doesn't work for detection (how many boxes should there be?). Instead:
1. Run the detector (with NMS) on all test images.
2. **For each category**, sort all its detections from highest to lowest score. Go down the list:
   * if a detection matches a ground-truth box with **IoU > 0.5**, it's a **true positive** (and that ground-truth box is used up);
   * otherwise it's a **false positive**.
3. After each detection, compute **precision** (what fraction of our detections so far were right) and **recall** (what fraction of the ground-truth boxes we've found).
4. **Average Precision (AP)** = the area under the precision–recall curve.
5. **mAP** = the average of AP over all categories.

**The lecture's example:** 5 dog detections, 3 ground-truth dogs. Going down the list from the highest score: match, match, no match, no match, match.

In [ ]:
matches = [True, True, False, False, True]          # detections sorted by score: 0.99, 0.95, 0.90, 0.5, 0.10
num_ground_truth = 3

true_positives = 0
precisions, recalls = [], []
for k, is_match in enumerate(matches, start=1):
    if is_match:
        true_positives += 1
    precisions.append(true_positives / k)
    recalls.append(true_positives / num_ground_truth)
    print(f"after {k} detections: precision = {true_positives}/{k} = {precisions[-1]:.2f}, recall = {true_positives}/3 = {recalls[-1]:.2f}")

The area under the curve: recall goes up in steps of 1/3, and each step happens at some precision. Adding up (precision × size of the recall step):

In [ ]:
def average_precision(matches, num_ground_truth):
    # Area under the precision-recall curve: sum of precision at each true positive x (1 / number of GT boxes).
    true_positives, area = 0, 0.0
    for k, is_match in enumerate(matches, start=1):
        if is_match:
            true_positives += 1
            area += (true_positives / k) * (1 / num_ground_truth)
    return area

print(f"dog AP = {average_precision(matches, 3):.3f}   (slides: 0.86, the same number rounded down)")
plt.figure(figsize=(4, 3))
plt.step(recalls, precisions, where="post", marker="o")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("precision-recall curve"); plt.ylim(0, 1.05)
plt.show()

**How do you get AP = 1.0?** Find **every** ground-truth box with IoU > 0.5, and rank **no** false positive above any true positive.

Then **mAP** averages over categories. The lecture's example: dog 0.86, cat 0.80, car 0.65 → **mAP@0.5 = 0.77**. And **"COCO mAP"** goes further: compute mAP at IoU thresholds 0.5, 0.55, 0.60, …, 0.95, and average them, which rewards boxes that are *precisely* placed.

In [ ]:
print(f"mAP@0.5 = {np.mean([0.86, 0.80, 0.65]):.2f}")

## 9. Making R-CNN fast

**Problem with "slow" R-CNN: ~2,000 separate CNN forward passes per image!**

### Fast R-CNN (Girshick, 2015): run the CNN *before* cropping
1. Run the **whole image** through a **backbone** CNN once (AlexNet, VGG, ResNet, …) to get image features.
2. For each region proposal, **crop and resize the features** (not the pixels).
3. A small **per-region network** predicts the class and box transform from each cropped feature.

Most of the computation happens in the backbone, which is now **shared** by all the overlapping proposals. (For example, with ResNet the last stage is the per-region network and the rest is the backbone.)

A rough cost comparison (in "units" of one CNN pass on a 224×224 crop):

In [ ]:
n_proposals = 2000
slow_rcnn = n_proposals * 1.0                     # one full CNN pass per proposal
image_area_ratio = (800 * 600) / (224 * 224)      # the backbone runs once on the whole (bigger) image
fast_rcnn = image_area_ratio * 0.9 + n_proposals * 0.1   # say 90% of the CNN is backbone, 10% per-region
print(f"slow R-CNN: about {slow_rcnn:,.0f} units of work per image")
print(f"fast R-CNN: about {fast_rcnn:,.0f} units  ({slow_rcnn / fast_rcnn:.0f}x less)")

### How to crop features: RoI Pool and RoI Align
The backbone shrinks the image (e.g., a 640×480 image becomes a 512×20×15 feature map, a factor of 32). Project the proposal onto the feature map by dividing its coordinates by 32, then cut it into a fixed grid (e.g., 7×7) and max-pool each cell. **Every region becomes the same size**, whatever its original size.

* **RoI Pool:** "snap" the box to the feature-map grid cells. Simple, but the features end up **slightly misaligned** with the box, and the subregions have different sizes.
* **RoI Align** (Mask R-CNN, 2017): **no snapping**. Sample the features at exact, evenly spaced points with **bilinear interpolation**, then max-pool.

Here's the snapping problem:

In [ ]:
proposal_in_image = np.array([100, 70, 330, 280])          # (x1, y1, x2, y2) in pixels
on_feature_map = proposal_in_image / 32                     # the backbone downsamples by 32
snapped = np.round(on_feature_map)                          # RoI Pool snaps to whole grid cells
print("proposal on the feature map:", on_feature_map)
print("after snapping (RoI Pool):  ", snapped, "  <- shifted by up to half a cell = 16 pixels!")

**Bilinear interpolation** (RoI Align): the feature at a point between grid cells is a **weighted average of the 4 nearest grid cells**, with more weight for closer cells. The lecture's example, the feature at $(x, y) = (6.5, 5.8)$:

$$f_{6.5,5.8} = f_{6,5}(0.5)(0.2) + f_{7,5}(0.5)(0.2) + f_{6,6}(0.5)(0.8) + f_{7,6}(0.5)(0.8)$$

(The point is halfway between columns 6 and 7, so each gets 0.5; it's 0.8 of the way from row 5 to row 6, so row 6 gets 0.8 and row 5 gets 0.2.)

In [ ]:
def bilinear(features, x, y):
    x0, y0 = int(np.floor(x)), int(np.floor(y))       # the grid cell up and to the left
    dx, dy = x - x0, y - y0                            # how far we are toward the next cell
    return (features[y0, x0] * (1 - dx) * (1 - dy) + features[y0, x0 + 1] * dx * (1 - dy)
            + features[y0 + 1, x0] * (1 - dx) * dy + features[y0 + 1, x0 + 1] * dx * dy)

features = np.arange(100, dtype=float).reshape(10, 10)   # a fake feature map: value = 10 * row + column
value = bilinear(features, 6.5, 5.8)
by_hand = features[5, 6] * 0.5 * 0.2 + features[5, 7] * 0.5 * 0.2 + features[6, 6] * 0.5 * 0.8 + features[6, 7] * 0.5 * 0.8
print(f"bilinear at (6.5, 5.8): {value:.2f}   by the formula: {by_hand:.2f}")

Because the result is a weighted sum, it's **differentiable**: in backprop, the gradient flows to each of the four grid cells in proportion to its weight (and even to the box coordinates).

### Faster R-CNN (Ren et al., 2015): learn the proposals too
After Fast R-CNN, the slowest part is the region-proposal heuristic (Selective Search on the CPU). **Faster R-CNN** replaces it with a **Region Proposal Network (RPN)** that predicts proposals from the backbone's features.

**How the RPN works:**
* Imagine an **anchor box** of a fixed size centered at **every point** of the feature map.
* At each point, predict: **is this anchor an object?** (a score) and a **box transform** from the anchor to the object.
* One size doesn't fit all objects, so use **K different anchors** (sizes and shapes) at each point.
* At test time, sort all the anchors by score and keep the top ~300 as proposals.

In [ ]:
def make_anchors(feature_h, feature_w, stride=32, sizes=(64, 128, 256), ratios=(0.5, 1.0, 2.0)):
    # One anchor of each size and aspect ratio, centered at every feature-map cell. Returns (x1, y1, x2, y2) boxes.
    anchors = []
    for row in range(feature_h):
        for col in range(feature_w):
            cx, cy = (col + 0.5) * stride, (row + 0.5) * stride        # the cell's center, in image pixels
            for size in sizes:
                for ratio in ratios:
                    w, h = size * np.sqrt(ratio), size / np.sqrt(ratio)  # same area, different shape
                    anchors.append((cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2))
    return np.array(anchors)

anchors = make_anchors(15, 20)                       # a 640x480 image -> 20x15 feature map
print(f"K = 9 anchors at each of 20 x 15 positions: {len(anchors):,} anchors")
print(f"RPN outputs: objectness {9} x 20 x 15, box transforms {4 * 9} x 20 x 15")

**Faster R-CNN is a two-stage detector**, trained jointly with **4 losses**:
* **Stage 1** (once per image): backbone + RPN. Losses: (1) anchor is object / not object; (2) transform from anchor to proposal.
* **Stage 2** (once per region): crop features (RoI Pool/Align), then (3) classify the proposal (background or an object class) and (4) predict the transform from proposal to final box.

### Single-stage detectors: YOLO, SSD, RetinaNet
**Do we really need the second stage?** A **single-stage** detector makes the RPN do everything: instead of "object / not object", each anchor directly predicts **one of C categories (or background)**, plus the box transform.

**YOLO ("You Only Look Once")**: divide the image into a grid (e.g., 7×7); in each cell, predict $B$ boxes (each with 4 coordinates + 1 confidence) and $C$ class scores:

In [ ]:
S, B, C = 7, 2, 20
print(f"YOLO output: {S} x {S} x (5 * {B} + {C}) = {S} x {S} x {5 * B + C} = {S * S * (5 * B + C)} numbers per image")

**Speed vs. accuracy** (Huang et al., 2017, and later Detectron2 results from the slides):
* **Two-stage** methods (Faster R-CNN) get the best accuracy, but are slower.
* **Single-stage** methods (SSD, YOLO) are much faster, but used to be less accurate; they've since improved a lot (e.g., RetinaNet).
* **Bigger backbones** help, but are slower. Feature Pyramid Networks (multi-scale features), longer training, and test-time augmentation also push the numbers up.
* Examples: Faster R-CNN + ResNet-101-FPN: 63 ms, 42.0 COCO mAP; Mask R-CNN + ResNeXt-152: 281 ms, 49.3 mAP.

## 10. Let's build a tiny detector!

Real detectors are big (and the lecture's advice is "**don't implement it yourself**": use Detectron2 or the TensorFlow Detection API). But a **tiny single-stage detector**, YOLO-style, is a great way to see all the pieces working together.

**Our dataset:** 32×32 images containing 1–3 shapes, either **squares** or **circles**, of different sizes. The detector must find each shape and say which kind it is.

In [ ]:
CLASSES = ["square", "circle"]

def make_image(rng):
    image = rng.normal(0, 0.05, size=(32, 32))
    boxes, labels, used_cells = [], [], set()
    for attempt in range(20):
        if len(boxes) == rng.integers(1, 4):
            break
        size = rng.integers(7, 13)
        x1, y1 = rng.integers(0, 32 - size, size=2)
        box = (int(x1), int(y1), int(x1 + size), int(y1 + size))
        cell = ((y1 + size // 2) // 8, (x1 + size // 2) // 8)        # which 8x8 grid cell holds the center
        if cell in used_cells or any(iou(box, b) > 0 for b in boxes):
            continue                                                  # avoid overlaps and shared cells
        label = rng.integers(2)
        if label == 0:
            image[y1:y1 + size, x1:x1 + size] += 1.0                  # a filled square
        else:
            yy, xx = np.mgrid[0:32, 0:32]
            center = (x1 + size / 2 - 0.5, y1 + size / 2 - 0.5)
            image[(xx - center[0])**2 + (yy - center[1])**2 <= (size / 2)**2] += 1.0   # a filled circle
        boxes.append(box); labels.append(int(label)); used_cells.add(cell)
    return image, boxes, labels

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax in axes:
    image, boxes, labels = make_image(rng)
    ax.imshow(image, cmap="gray")
    for (x1, y1, x2, y2), label in zip(boxes, labels):
        ax.add_patch(patches.Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, color="lime" if label == 0 else "orange"))
    ax.axis("off")
plt.suptitle("training images with ground-truth boxes (green = square, orange = circle)", fontsize=9)
plt.show()

### The design (YOLO-style)
* Divide the image into a **4×4 grid** of 8×8-pixel cells.
* Each cell is **responsible** for the object whose **center** falls in it.
* For each cell, a small network looks at a **24×24 window** around the cell (the cell plus its neighbors, so it can see whole objects) and predicts **7 numbers**:
  * 1 **objectness** score (is there an object centered here?),
  * 2 **class** scores (square vs. circle),
  * 4 **box** numbers: the center's position within the cell, $(t_x, t_y)$ in [0, 1], and the size as $(\log(w/8), \log(h/8))$, the same "log-space" trick as R-CNN.

**Step 1: the training targets for one image.**

In [ ]:
def make_targets(boxes, labels):
    # For each of the 16 cells: [has object, class, tx, ty, tw, th].
    targets = np.zeros((4, 4, 6))
    for (x1, y1, x2, y2), label in zip(boxes, labels):
        w, h = x2 - x1, y2 - y1
        cx, cy = x1 + w / 2, y1 + h / 2
        row, col = int(cy // 8), int(cx // 8)
        targets[row, col] = [1, label, (cx - col * 8) / 8, (cy - row * 8) / 8, np.log(w / 8), np.log(h / 8)]
    return targets

image, boxes, labels = make_image(np.random.default_rng(1))
print("boxes:", boxes, " labels:", [CLASSES[l] for l in labels])
t = make_targets(boxes, labels)
for row in range(4):
    for col in range(4):
        if t[row, col, 0] == 1:
            print(f"cell ({row}, {col}) is responsible for a {CLASSES[int(t[row, col, 1])]}: targets {t[row, col, 2:].round(2)}")

**Step 2: the input for each cell**: the 24×24 window around it (padding the image with zeros at the borders).

In [ ]:
def cell_windows(image):
    # The 24x24 window around each of the 16 cells, flattened: shape (16, 576).
    padded = np.pad(image, 8)
    windows = []
    for row in range(4):
        for col in range(4):
            windows.append(padded[row * 8:row * 8 + 24, col * 8:col * 8 + 24].ravel())
    return np.array(windows)

print("inputs for one image:", cell_windows(image).shape, "(16 cells, 24*24 pixels each)")

**Step 3: build a training set** of 2,000 images (32,000 cells):

In [ ]:
def make_dataset(n_images, seed):
    rng = np.random.default_rng(seed)
    images, all_boxes, all_labels, inputs, targets = [], [], [], [], []
    for _ in range(n_images):
        image, boxes, labels = make_image(rng)
        images.append(image); all_boxes.append(boxes); all_labels.append(labels)
        inputs.append(cell_windows(image))
        targets.append(make_targets(boxes, labels).reshape(16, 6))
    return images, all_boxes, all_labels, np.vstack(inputs), np.vstack(targets)

_, _, _, X_train, T_train = make_dataset(2000, seed=10)
print(f"training cells: {len(X_train):,}  ({int(T_train[:, 0].sum()):,} contain an object center)")

**Step 4: the network and its multitask loss.** A 2-layer network (576 → 128 → 7), and three losses added together (the "weighted sum" from Section 2):
* **objectness:** binary cross-entropy, for **every** cell;
* **class:** softmax cross-entropy, only for cells that **have** an object;
* **box:** L2, only for cells that have an object (weighted by 5, as YOLO does, because box errors matter a lot).

For each part, the gradient of the loss with respect to the network's outputs is simple (Lecture 3): probability minus target for the two classification losses, and $2\times$(prediction − target) for L2.

In [ ]:
def init_detector(seed=0):
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(0, np.sqrt(2 / 576), (576, 128)), "b1": np.zeros(128),
            "W2": rng.normal(0, np.sqrt(2 / 128), (128, 7)), "b2": np.zeros(7)}

def detector_forward(params, X):
    hidden = np.maximum(0, X @ params["W1"] + params["b1"])
    return hidden, hidden @ params["W2"] + params["b2"]          # 7 outputs per cell

def detector_loss(outputs, targets, box_weight=5.0):
    N = len(outputs)
    has_object = targets[:, 0] == 1
    d_out = np.zeros_like(outputs)
    # objectness (output 0): binary cross-entropy on every cell
    p_obj = 1 / (1 + np.exp(-outputs[:, 0]))
    obj_loss = -np.mean(targets[:, 0] * np.log(p_obj + 1e-9) + (1 - targets[:, 0]) * np.log(1 - p_obj + 1e-9))
    d_out[:, 0] = (p_obj - targets[:, 0]) / N
    # class (outputs 1-2): softmax cross-entropy, only where there's an object
    scores = outputs[has_object, 1:3]
    probs = np.exp(scores - scores.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
    classes = targets[has_object, 1].astype(int)
    class_loss = -np.sum(np.log(probs[np.arange(len(classes)), classes] + 1e-9)) / N
    probs[np.arange(len(classes)), classes] -= 1
    d_out[has_object, 1:3] = probs / N
    # box (outputs 3-6): L2, only where there's an object
    box_error = outputs[has_object, 3:7] - targets[has_object, 2:6]
    box_loss = box_weight * np.sum(box_error ** 2) / N
    d_out[has_object, 3:7] = box_weight * 2 * box_error / N
    return obj_loss + class_loss + box_loss, d_out

**Step 5: train** (backprop through the 2-layer network, as in Lecture 5, with SGD + momentum):

In [ ]:
params = init_detector()
velocity = {name: np.zeros_like(value) for name, value in params.items()}
rng = np.random.default_rng(0)
losses = []
for step in range(3000):
    batch = rng.integers(0, len(X_train), 256)
    hidden, outputs = detector_forward(params, X_train[batch])
    loss, d_out = detector_loss(outputs, T_train[batch])
    grads = {"W2": hidden.T @ d_out, "b2": d_out.sum(0)}
    d_hidden = (d_out @ params["W2"].T) * (hidden > 0)
    grads["W1"] = X_train[batch].T @ d_hidden
    grads["b1"] = d_hidden.sum(0)
    for name in params:
        velocity[name] = 0.9 * velocity[name] - 0.05 * grads[name]
        params[name] += velocity[name]
    losses.append(loss)
print(f"loss: {np.mean(losses[:50]):.3f} at the start -> {np.mean(losses[-50:]):.3f} at the end")

**Step 6: turn the outputs into boxes.** For each cell with objectness > 0.5, decode its box (undo the targets' encoding), and give it a **score** = P(object) × P(class). Then apply **NMS**.

In [ ]:
def detect(params, image, threshold=0.5):
    _, outputs = detector_forward(params, cell_windows(image))
    detections = []                                           # (box, class, score)
    for cell in range(16):
        row, col = cell // 4, cell % 4
        p_obj = 1 / (1 + np.exp(-outputs[cell, 0]))
        if p_obj < threshold:
            continue
        class_probs = np.exp(outputs[cell, 1:3]) / np.exp(outputs[cell, 1:3]).sum()
        label = int(np.argmax(class_probs))
        tx, ty, tw, th = outputs[cell, 3:7]
        cx, cy = col * 8 + tx * 8, row * 8 + ty * 8           # undo the encoding from make_targets
        w, h = 8 * np.exp(tw), 8 * np.exp(th)
        detections.append(((cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), label, p_obj * class_probs[label]))
    # non-max suppression, separately for each class
    final = []
    for label in range(2):
        same_class = [d for d in detections if d[1] == label]
        keep = nms([d[0] for d in same_class], np.array([d[2] for d in same_class]), threshold=0.5) if same_class else []
        final += [same_class[i] for i in keep]
    return final

In [ ]:
test_images, test_boxes, test_labels, _, _ = make_dataset(300, seed=99)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
for ax, image in zip(axes, test_images):
    ax.imshow(image, cmap="gray")
    for (x1, y1, x2, y2), label, score in detect(params, image):
        ax.add_patch(patches.Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, color="lime" if label == 0 else "orange", lw=1.5))
        ax.text(x1, y1 - 1, f"{score:.2f}", color="yellow", fontsize=7)
    ax.axis("off")
plt.suptitle("test images: predicted boxes (green = square, orange = circle) and scores", fontsize=9)
plt.show()

**Step 7: evaluate with mAP**, exactly as in Section 8: for each class, sort all detections by score, match each to an unused ground-truth box with IoU > 0.5, and compute AP.

In [ ]:
def evaluate_ap(params, images, all_boxes, all_labels, label, iou_threshold=0.5):
    detections = []                                     # (score, image index, box)
    num_gt = 0
    for i, image in enumerate(images):
        num_gt += sum(1 for l in all_labels[i] if l == label)
        for box, l, score in detect(params, image):
            if l == label:
                detections.append((score, i, box))
    detections.sort(key=lambda d: -d[0])                # highest score first
    used = set()
    matches = []
    for score, i, box in detections:
        best_iou, best_gt = 0, None
        for j, (gt_box, gt_label) in enumerate(zip(all_boxes[i], all_labels[i])):
            if gt_label == label and (i, j) not in used and iou(box, gt_box) > best_iou:
                best_iou, best_gt = iou(box, gt_box), j
        if best_iou > iou_threshold:
            used.add((i, best_gt))                      # this ground-truth box is now used up
            matches.append(True)
        else:
            matches.append(False)
    return average_precision(matches, num_gt)

aps = [evaluate_ap(params, test_images, test_boxes, test_labels, label) for label in range(2)]
for name, ap in zip(CLASSES, aps):
    print(f"AP for {name}s: {ap:.2f}")
print(f"mAP@0.5 = {np.mean(aps):.2f}")

And a stricter, COCO-style score: the average mAP over IoU thresholds from 0.5 to 0.95. It rewards boxes that are placed **precisely**:

In [ ]:
thresholds = np.arange(0.5, 0.96, 0.05)
maps = [np.mean([evaluate_ap(params, test_images, test_boxes, test_labels, label, t) for label in range(2)]) for t in thresholds]
for t, m in zip(thresholds, maps):
    print(f"mAP@{t:.2f} = {m:.2f}")
print(f"COCO-style mAP (average) = {np.mean(maps):.2f}")

**What we see:** at IoU 0.5 our tiny detector finds most shapes and names them correctly; as the IoU threshold gets stricter, mAP drops, because some boxes are a pixel or two off. Real detectors show the same pattern (e.g., the slides' example: mAP@0.5 = 0.77 but COCO mAP = 0.4).

## 11. Detection without anchors: a glimpse
* **CornerNet** (2018): represent each box by its **top-left and bottom-right corners**, predicted as heatmaps.
* **DETR** (2020): use a **Transformer** to output a set of boxes directly, matched to the ground truth one-to-one. No anchors, no RPN, no NMS!

More on these next lecture.

## 12. In practice (optional, run in Colab)
**"Object detection is hard! Don't implement it yourself."** Use a library: **Detectron2** (PyTorch) or torchvision's detectors.

In [ ]:
# [PyTorch] Needs: pip install torch torchvision (downloads pretrained weights, about 160 MB)
import torch, torchvision
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")   # pretrained on COCO
model.eval()
image = torch.rand(3, 480, 640)                         # use a real photo here (values in 0..1)
with torch.no_grad():
    result = model([image])[0]
print("boxes:", result["boxes"].shape, " labels:", result["labels"][:5], " scores:", result["scores"][:5])
print("NMS and IoU are built in:", torchvision.ops.nms, torchvision.ops.box_iou)

## Summary
* **Detection** = a category + a box for every object. Hard because the number of objects varies.
* **Single object:** classification head + box regression head, trained with a **multitask loss**.
* **R-CNN:** region proposals → CNN on each crop → class + **box transform** (shift relative to size, scale in log space).
* **Fast R-CNN:** run the backbone **once**, then crop **features** with **RoI Pool** (snaps to the grid) or **RoI Align** (bilinear, no snapping).
* **Faster R-CNN:** a **Region Proposal Network** with **anchors** learns the proposals. Two stages, 4 losses.
* **Single-stage** (YOLO, SSD, RetinaNet): each anchor/grid cell predicts the class directly. Faster, nearly as accurate today.
* **IoU** compares boxes; **NMS** removes duplicates; **mAP** evaluates detectors (COCO averages over IoU 0.5 to 0.95).

### Check your understanding
1. Why can't a classification network with one extra "box" output detect several objects?
2. Why does Fast R-CNN run so much faster than R-CNN?
3. What's the difference between RoI Pool and RoI Align?
4. Your detector outputs 5 overlapping boxes on one dog. What fixes that?
5. How do you get an AP of 1.0?

<details><summary>Answers</summary>

1. Each image has a different number of objects, but a fixed output layer always gives the same number of outputs.
2. The expensive backbone runs once per image instead of once per proposal; only a small per-region network runs per proposal.
3. RoI Pool snaps the box to feature-map grid cells (misaligned); RoI Align samples features at exact points with bilinear interpolation.
4. Non-max suppression.
5. Find every ground-truth box with IoU > 0.5, and never rank a false positive above a true positive.
</details>

**Next:** Lecture 16, detection + segmentation.